# Khmer OCR selected demo
Click **Runtime → Run all** in Google Colab. No GPU or Hugging Face login is required for the public model.
These are selected successful examples, not an independent accuracy benchmark. The notebook performs real inference and shows the expected answer and prediction.


In [ ]:
%pip -q install torch pillow numpy huggingface_hub
from huggingface_hub import hf_hub_download
from pathlib import Path
import sys, json, zipfile
repo = "Seoul12/khmer-ocr-architecture3-best"
checkpoint = hf_hub_download(repo, "best.pt")
module = hf_hub_download(repo, "inference.py")
sys.path.insert(0, str(Path(module).parent))
from inference import Recognizer
model = Recognizer(checkpoint)
print("Loaded saved checkpoint, step:", model.step)
archive = hf_hub_download("Seoul12/khmer-ocr-selected-demo", "demo_dataset.zip", repo_type="dataset")
with zipfile.ZipFile(archive) as z:
    z.extractall("selected_demo")
manifest_path = Path("selected_demo/demo/manifest.json")
rows = json.loads(manifest_path.read_text())


In [ ]:
from IPython.display import display
from PIL import Image
correct = 0
for row in rows:
    path = manifest_path.parent / row["image"]
    prediction = model.predict(path)
    match = prediction == row["answer"]
    correct += match
    display(Image.open(path))
    print("ANSWER:", row["answer"])
    print("PREDICTION:", prediction)
    print("RIGHT" if match else "WRONG")
print(f"Selected examples: {correct}/{len(rows)} exact matches. This is not benchmark accuracy.")


## Try your own cropped text-line image
Printed text is generally more reliable than handwriting. Results on new images can contain errors.

In [ ]:
# Set this to True only when you want to upload a new cropped text line.
TRY_YOUR_IMAGE = False
if TRY_YOUR_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded:
        display(Image.open(filename))
        print("PREDICTION:", model.predict(filename))
